In [3]:
# import necessary libraries
import requests
from bs4 import BeautifulSoup 
from collections import defaultdict
import json
import sqlite3
import re
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [15]:
# creates dictionary of dictionaries --> nested dictionaries
dataDict = defaultdict(dict) 

## Websites I unsuccessfully tried to webscrap
# website = "https://www.myrecipes.com/favorites#/collection/53424894"
# website = "https://www.foodnetwork.com/recipes/dinner"

links = []
website = f"https://www.nutrition.gov/recipes/search"
webpage = requests.get(website) # API call to website
soup = BeautifulSoup(webpage.content, 'lxml')

# Download into a text file


li = soup.find_all('li', class_="usa-pagination__item usa-pagination__arrow usa-pagination__page-no")
pages = [l.a.get('href') for l in li]

# Collects recipe urls from all pages on www.nutrition/gov
for p in pages:
    website = "https://www.nutrition.gov/recipes/search" + p
    webpage = requests.get(website) # API call to website
    soup = BeautifulSoup(webpage.content, 'lxml')
    links = links + soup.find_all('a', rel="bookmark")

# Webscraps all recipe urls for recipe name and ingredients
for i in range(len(links)): # rel="bookmark" helps filter out the recipe links
    recipe_link = "https://www.nutrition.gov" + links[i].get('href')
    recipe_page = requests.get(recipe_link) # API call to recipe website
    recipe_soup = BeautifulSoup(recipe_page.content, 'lxml')
    recipe_name = recipe_soup.h1.span.string # Would "get_text" work???
    span = recipe_soup.find_all('span', class_ = "ingredient-name")
    ingr_list = [re.sub(r"(\(.+\))", "",ingr.string) for ingr in span] # list of ingredients for each recipe
    ingr_csv = ", ".join(ingr_list) # convert list to comma-separated values (CSV)
    
    # store data into separate dictionaries
    dataDict['name'][i] = recipe_name 
    dataDict['url'][i] = recipe_link 
    dataDict['ingredients'][i] = ingr_csv
    
print("Total number of recipes:", len(dataDict['name']))

142


In [ ]:
'''Created a json file that holds the recipes data.'''
filename = "recipes.json"
with open(filename, 'w') as f:
    json.dump(dataDict, f) # writes data into a json file
    print("JSON file created.")

JSON file created.


In [ ]:
# try:
#     recipes = pd.read_json('recipes.json')
# except ValueError as e:
#     print("ValueError:", e)
    
# with open("recipeitems-latest.json") as f:
#     line = f.readline()
    
# pd.read_json(line).shape

In [ ]:
# for tag in div:
#     print(tag.find_all('span', class_ ="quantity-unit"))
#     print(tag.find_all('span', class_="ingredient-name"))


In [ ]:
'''This code stores recipe data JSON file into a SQL database.'''

# with open(filename) as f:
#     data = json.load(f) # stores data from json file into data object
    
#     # connect
#     conn = sqlite3.connect('recipes.db')
#     c = conn.cursor()
    
#     # create table
#     c.execute(
#         """DROP TABLE IF EXISTS recipes""")
#     c.execute(
#         """CREATE TABLE recipes (
#             id INTEGER NOT NULL PRIMARY KEY, 
#             name TEXT, 
#             url TEXT,
#             ingredients TEXT
#             )""")
    
#     for i in range(len(data['name'])):
# #         print(data['name'][i], data['url'][i], data['ingredients'][i])
#         c.execute("""INSERT INTO recipes VALUES (?, ?, ?, ?)""", 
#                   (i+1,
#                    data['name'][i], 
#                    data['url'][i],
#                    data['ingredients'][i]))
        
#     print("Database created.")
#     conn.commit()
#     conn.close()

Database created.


In [ ]:
# View recipes SQL Database
# conn = sqlite3.connect('recipes.db')
# c = conn.cursor()
# for record in c.execute("""SELECT * FROM recipes""") :
#     print(record)
# conn.close()

(1, 'Three Sisters Salad', 'https://www.nutrition.gov/recipes/three-sisters-salad', 'sugar, vinegar , olive oil, corn , black beans , zucchini , squash , onion , bell peppers ')
(2, 'Maple Brined Pork Chops', 'https://www.nutrition.gov/recipes/maple-brined-pork-chops', 'water, maple syrup, salt, peppercorn, bay leaves , ice cube, pork chop , vegetable oil, peach ')
(3, 'Sheet Pan Sausage and Veggie Bake', 'https://www.nutrition.gov/recipes/sheet-pan-sausage-and-veggie-bake', 'Nonstick cooking spray, bell pepper and onion mix , sausage , butternut squash , black pepper, garlic powder, Italian seasoning')
(4, 'Quick Lasagna', 'https://www.nutrition.gov/recipes/quick-lasagna', ' ground beef , onion , garlic , tomato sauce , parsley , oregano, basil, cottage cheese , mozzarella cheese , lasagna noodle , grated parmesan cheese')
(5, '5 A Day Salad', 'https://www.nutrition.gov/recipes/5-day-salad', 'spinach leaves , romaine lettuce, bell pepper , cherry tomatoes, broccoli , cauliflower , squ

In [3]:
!curl -O http://openrecipes.s3.amazonaws.com/recipeitems-latest.json.gz
!gunzip recipeitems-latest.json.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100    20  100    20    0     0     96      0 --:--:-- --:--:-- --:--:--    96


In [55]:
website = "www.themealdb.com/api/json/v1/1/search.php?s=Arrabiata"
page = requests.get(website) # API call to website
soup = BeautifulSoup(page.content, 'lxml')

# try:
#     recipes = pd.read_json('recipeitems-latest.json')
# except ValueError as e:
#     print("ValueError:", e)
    
# with open("recipeitems-latest.json") as f:
#     line = f.readline()
    
# pd.read_json(line).shape

ValueError: Expected object or value


ValueError: Expected object or value

In [ ]:
# read the entire file into a Python array
with open('recipeitems-latest.json', 'r') as f:
    # Extract each line
    data = (line.strip() for line in f)
    # Reformat so each line is the element of a list
    data_json = "[{0}]".format(','.join(data))
# read the result as a JSON
recipes = pd.read_json(data_json)
recipes.shape

In [ ]:
recipes.iloc[0]